In [ ]:
!pip install sentence-transformers torch numpy

In [ ]:
import json

data = []
try:
    with open("/content/knowledge_chunks.json","r") as f:
        data = json.load(f)
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}. Attempting to parse as JSON Lines.")
    # If standard json.load fails, try loading line by line as JSON Lines
    temp_data = []
    with open("/content/knowledge_chunks.json","r") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line: # Skip empty lines
                continue
            try:
                temp_data.append(json.loads(line))
            except json.JSONDecodeError as line_e:
                print(f"Skipping malformed JSON line {line_num}: {line} - Error: {line_e}")
                # Optionally, you could log this or decide to stop parsing
                continue
    data = temp_data
    if not data:
        print("Failed to load any data using JSON Lines parsing.")
    else:
        print(f"Successfully loaded {len(data)} objects using JSON Lines parsing.")

In [ ]:
len(data)

31355

In [ ]:
texts=[i['text'] for i in data]
print(texts[:10])
len(texts)

['Formulation arjunarishta contains the plant Terminalia arjuna.', 'Formulation arjunarishta contains the plant Vitis vinifera.', 'Formulation arjunarishta contains the plant Woodfordia fruticosa.', 'Formulation arjunarishta contains the plant Madhuca longifolia.', 'Formulation draksharishta contains the plant Vitis vinifera.', 'Formulation draksharishta contains the plant Woodfordia fruticosa.', 'Formulation draksharishta contains the plant Cinnamomum tamala.', 'Formulation draksharishta contains the plant Piper nigrum.', 'Formulation draksharishta contains the plant Mesua ferrea.', 'Formulation abhayarishta contains the plant Terminalia chebula.']


31355

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/980 [00:00<?, ?it/s]

(31355, 768)


In [ ]:
import numpy as np

np.save("/content/embeddings.npy", embeddings)


In [ ]:
!pip install ollama tqdm faiss-gpu-cu11

  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 66.6 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
import ollama
from tqdm import tqdm

def embed_text(text):
    response = ollama.embeddings(
        model="phi3:mini",
        prompt=text
    )
    return response["embedding"]

embeddings = []

for text in tqdm(texts, desc="Embedding chunks"):
    emb = embed_text(text)
    embeddings.append(emb)

embeddings = np.array(embeddings, dtype="float32")

dimension = embeddings.shape[1]
print("Embedding dimension:", dimension)

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors indexed:", index.ntotal)

faiss.write_index(index, "ayurveda_phi.index")
np.save("embeddings_phi.npy", embeddings)

print("FAISS index saved to ayurveda.index")


Embedding chunks:   0%|          | 0/31355 [00:00<?, ?it/s]


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
!python -m pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
!pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.4 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


# part 2

In [ ]:
import json
from langchain_core.documents import Document

def load_chunks(path="chunks.json"):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    docs = []
    for item in raw:
        docs.append(
            Document(
                page_content=item["text"],
                metadata=item["metadata"]
            )
        )
    return docs

documents = load_chunks("/content/chunks.json")
print(f"Loaded {len(documents)} chunks")


Loaded 31355 chunks


In [ ]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = []
with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)
    for c in chunks:
        texts.append(c["text"])

embeddings = model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

np.save("embeddings_langchain.npy", embeddings)

Batches:   0%|          | 0/980 [00:00<?, ?it/s]

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/tmp/ipython-input-3055314890.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import numpy as np


np.save("embeddings_langchain.npy", embeddings)
print("Embeddings saved to embeddings_langchain.npy")

TypeError: float() argument must be a string or a real number, not 'HuggingFaceEmbeddings'

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.7 MB/s eta 0:00:00


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

vectorstore.save_local("ayurveda_faiss")


In [ ]:
vectorstore = FAISS.load_local(
    "ayurveda_faiss",
    embeddings,
    allow_dangerous_deserialization=True
)

In [ ]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="phi3:mini",
    temperature=0.2
)


/tmp/ipython-input-1405617171.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 15}
)


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an expert Ayurvedic knowledge system.

Use ALL provided information:
- text
- metadata
- relationships between formulations, plants, phytochemicals

Rules:
- You may derive NEW information by logically combining facts
- If a formulation contains a plant, and the plant contains phytochemicals,
  you may infer indirect associations
- Cite facts explicitly
- If unsure, say so clearly

Context:
{context}

Question:
{question}

Answer with clear reasoning:
"""
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)


In [ ]:
query = "Which phytochemicals are indirectly associated with arjunarishta?"

result = qa.invoke(query)

print("ANSWER:\n", result["result"])

print("\nSOURCES:")
for doc in result["source_documents"]:
    print("-", doc.page_content)


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f21d7d87d40>: Failed to establish a new connection: [Errno 111] Connection refused'))

# part 3

In [ ]:
import json
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# -------- CONFIG --------
CHUNKS_PATH = "knowledge_chunks2.json"
INDEX_PATH = "index.faiss"
META_PATH = "index.pkl"
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BATCH_SIZE = 64
# ------------------------

print("Loading chunks...")
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = []
metadatas = []

for c in chunks:
    # Include metadata in embedding context
    meta_str = " | ".join(f"{k}: {v}" for k, v in c["metadata"].items())
    combined = f"{c['text']} || {meta_str}"
    texts.append(combined)
    metadatas.append(c)

print(f"Loaded {len(texts)} chunks")

print("Loading embedding model...")
model = SentenceTransformer(MODEL_NAME)

print("Embedding...")
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

print("Building FAISS index...")
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, INDEX_PATH)

with open(META_PATH, "wb") as f:
    pickle.dump(metadatas, f)

print("Index built successfully!")


Loading chunks...
Loaded 32843 chunks
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding...


Batches:   0%|          | 0/514 [00:00<?, ?it/s]

Building FAISS index...
Index built successfully!


In [ ]:
np.save("embeddings.npy", embeddings)